# Fase 2: Ingeniería de características

In [6]:
# Importar dependencias.
import constants
import numpy as np
import pandas as pd
import warnings
from util import csv_util

# No mostrar mensajes de advertencia.
warnings.filterwarnings('ignore')

## 1. Introducción

En esta fase, se obtendrá el conjunto de datos final que será procesado por los diferentes modelos de inteligencia artificial. Para ello, se seleccionarán las características más relevantes, es decir, que cumplan tres propiedades:
- **Informativa**: aporta valor para que el modelo aprenda o reconozca patrones.
- **Discriminativa**: es útil para establecer diferencias entre los registros del conjunto de entrenamiento.
- **No redundante**: no existe otra característica que dé la misma información.

También se generarán nuevas características a partir de las actuales si dichas transformaciones aportan más información. El objetivo es aumentar la precisión para que el modelo generalice mejor y, así, evitar el sobreajuste.

## 2. Selección y extracción de características

Por un lado, dado que el conteo de tráfico está asociado a un carril, se seleccionarán características identificativas de este elemento:

- **Fecha** sin hora.
- **Ciudad**.
- Identificador de la **sección**.
- Identificador de la **calzada**, que tiene uno o más carriles en una dirección concreta.
- Identificador de la **dirección** (sentido de circulación), representado con dos valores (1 o 2).
- Identificador secuencial del **carril**, empezando por el más lento o, dicho de otro modo, el más pegado al arcén. Éste se representa con el 1.

Por otro lado, se generarán nuevas características asociadas con el deterioro del pavimento de las carreteras. Éstas están asociadas con el tráfico de vehículos pesados y datos meteorológicos:
- **ESAL** (_Equivalent Single Axle Load_): esta medida permite transformar las diferentes cargas de vehículos (número y distribución de ejes, y peso) que circulan por una carretera en un número equivalente. El tráfico ligero es despreciable. Sin embargo, no se dispone de la configuración de cada vehículo pesado contabilizado. Por ello, se aplicará un factor de tráfico por carril. Los vehículos más pesados (camiones o autobuses) suelen circulan por el carril lento por limitación de velocidad. Los más livianos (furgonetas) suelen circular por otros carriles.
- **Estrés térmico**: impacto de temperaturas extremas por calor y el índice ultravioleta.
- **Estrés por oscilación térmica**: impacto de la diferencia de la temperatura máxima y mínima.
- **Estrés hídrico**: impacto de las lluvias.

Por último, la temperatura de la superficie del asfalto no es la misma que la del aire. Para calcularla, se aplicará la siguiente fórmula, obtenida del artículo científico _Model to Predict Pavement Temperature Proﬁle: Development and Validation_ de Journal Of Transportation Engineering.

$T_{p\,\text{max}} = 2,78752 + 0,6861 \cdot T_{a\,\text{max}} + (5,6736 \times 10^{-4}) \cdot R_s - 27,8739 \cdot P_d$

$T_{p\,\text{min}} = -1,2097 + 0,6754 \cdot T_{a\,\text{min}} + (3,7642 \times 10^{-4}) \cdot R_s + 7,2043 \cdot P_d$

* **$T_{p\,\text{max}}$**: temperatura máxima del pavimento en grados Celsius.
* **$T_{p\,\text{min}}$**: temperatura mínima del pavimento en grados Celsius.
* **$T_{a\,\text{max}}$**: temperatura máxima ambiental en grados Celsius.
* **$T_{a\,\text{min}}$**: temperatura mínima ambiental en grados Celsius.
* **$R_s$**: radiación solar diaria acumulada en kilojulios por metro cuadrado y día.
* **$P_d$**: profundidad del pavimento en metros, pero su valor será 0 para cálculos en la superficie.

In [7]:
def load_preprocessed_data(city):
    # Leer ficheros CSV.
    df_traffic = csv_util.read(f'model/SAITS/result/{city}_traffic.csv',
                               usecols=['date', 'city', 'section', 'roadway', 'direction', 'lane', 'heavy_traffic'],
                               dtype={'city':str, 'section':int, 'roadway':int, 'direction':int, 'lane':int, 'heavy_traffic':float},
                               date_formats=[('date', '%Y-%m-%d')])
    df_weather = csv_util.read(f'dataset/raw_data/weather/{city}_weather.csv',
                               usecols=['datetime', 'tempmax', 'tempmin', 'precip', 'solarenergy', 'uvindex'],
                               dtype={'tempmax':float, 'tempmin':float, 'precip':float, 'solarenergy':float, 'uvindex':float},
                               date_formats=[('datetime', '%Y-%m-%d')])

    # Solo registros meteorológicos para los años de estudio.
    df_weather = df_weather[df_weather['datetime'].between(constants.START_DATE, constants.END_DATE)]

    # Cambiar unidad a grados Celsius.
    df_weather['tempmax'] = (df_weather['tempmax'] - 32) / 1.8
    df_weather['tempmin'] = (df_weather['tempmin'] - 32) / 1.8

    # Cambiar unidad a kilojulios.
    df_weather['solarenergy'] = df_weather['solarenergy'] * 1000

    # Calcular temperatura mínima y máxima del pavimento.
    df_weather['min_pavement_temperature'] = -1.2097 + (0.6754 * df_weather['tempmin']) + (0.00037642 * df_weather['solarenergy'])
    df_weather['max_pavement_temperature'] = 2.78752 + (0.6861 * df_weather['tempmax']) + (0.00056736 * df_weather['solarenergy'])
    df_weather = df_weather.rename(columns={'datetime':'date', 'precip':'precipitation', 'uvindex':'uv_index'})

    # Eliminar características innecesarias.
    df_weather = df_weather.drop(columns=['tempmax', 'tempmin', 'solarenergy'])

    # Unir ambos dataframes.
    df_full = pd.merge(df_traffic, df_weather, on=['date'], how='inner')

    return df_full

def load_sections(city):
    df_sections = csv_util.read(f'dataset/raw_data/sections/{city}_sections.csv',
                                usecols=['section_id', 'dir_1', 'dir_2', 'lanes', 'description'],
                                dtype={'section_id':int, 'dir_1':int, 'dir_2':int, 'lanes':int, 'description':str})

    return df_sections

def obtain_total_lanes_by_roadway(city):
    total_lanes_by_roadway = {}
    df_sections = load_sections(city)

     # Obtener relación entre número de carriles y calzada.
    for _, row in df_sections.iterrows():
        total_lanes_by_roadway[row['dir_1']] = row['lanes']
        total_lanes_by_roadway[row['dir_2']] = row['lanes']

    return total_lanes_by_roadway

def obtain_truck_factor_by_lane(lane, total_lanes):
    if lane == 1:
        # Carril lento (más camiones pesados).
        return constants.SLOW_LANE_TRUCK_FACTOR
    elif lane == total_lanes:
        # Carril rápido (furgonetas o pequeños camiones).
        return constants.FAST_LANE_TRUCK_FACTOR
    else:
        # Carril central para circunvalaciones.
        return constants.CENTER_LANE_TRUCK_FACTOR

def obtain_road_damage(df_full, total_lanes_by_roadway):
    # Calcular ESAL en función del carril.
    df_full['total_lanes'] = df_full['section'].map(total_lanes_by_roadway)
    df_full['esal'] = df_full.apply(lambda row: row['heavy_traffic'] * obtain_truck_factor_by_lane(row['lane'], row['total_lanes']), axis=1)

    # Calcular factores ambientales (calor extremo, fotooxidación por UV, oscilación térmica y estrés hídrico).
    environmental_stress = (np.maximum(0, df_full['uv_index'] * ((df_full['max_pavement_temperature'] - constants.EXTREME_HEAT) / 10.0)) ** 1.5606
                            + (df_full['max_pavement_temperature'] - df_full['min_pavement_temperature']) ** 1.35
                            + np.log1p(df_full['precipitation']))

    # Acumulado diario.
    df_full['total_stress'] = df_full['esal'] * environmental_stress

    # Eliminar características innecesarias y ordenar.
    df_full = df_full.drop(columns=['total_lanes', 'esal'])
    df_full = df_full.sort_values(['date', 'city', 'section', 'roadway', 'direction', 'lane'])

    return df_full

Finalmente, se generará el conjunto de datos final para cada ciudad.

In [8]:
csv_util.delete('dataset/processed_data/dataset.csv')

for city in constants.CITIES:
    # Cargar datos preprocesados.
    df_full = load_preprocessed_data(city)

    # Añadir daño acumulado diario.
    df_full = obtain_road_damage(df_full, obtain_total_lanes_by_roadway(city))

    # Guardar el conjunto de datos por ciudad y conjuntamente.
    csv_util.save(df_full, f'dataset/processed_data/{city}_dataset.csv', mode='w')
    csv_util.save(df_full, 'dataset/processed_data/dataset.csv', mode='a')